# Revela — Notebook 05: BCN20000 Cancer-Risk Splits

**Issue:** #118 — D3.3: Rebuild BCN20000 processed splits for cancer-risk taxonomy  
**Status:** Closed  
**Depends on:** #117 (D3.2) — 4-class cancer-risk taxonomy approved  
**Script:** `src/data/prepare_bcn20000_cancer_risk.py`  
**Config:** `config/bcn20000_cancer_risk_config.yaml`  

### Purpose
Document the BCN20000 split rebuild for the 4-class cancer-risk taxonomy. This notebook covers:
- What changed from the old 3-class splits
- How the 4-class label mapping is applied
- Lesion-level leakage safety verification
- Output split counts and class distribution

### Deliverables
- `data/processed/bcn20000_cancer_risk/train.csv` — 12,352 rows
- `data/processed/bcn20000_cancer_risk/val.csv` — 2,628 rows
- `data/processed/bcn20000_cancer_risk/test.csv` — 2,659 rows
- `data/processed/bcn20000_cancer_risk/master_metadata.csv` — 17,639 rows
- `docs/model/bcn20000_cancer_risk_split_summary.md`

> Old 3-class splits in `data/processed/bcn20000/` are untouched.

## Block 1 — What Changed from the Old 3-Class Splits

| Item | Old (3-class) | New (cancer-risk) |
|---|---|---|
| Script | `src/data/prepare_bcn20000.py` | `src/data/prepare_bcn20000_cancer_risk.py` |
| Config | `config/bcn20000_config.yaml` | `config/bcn20000_cancer_risk_config.yaml` |
| Output dir | `data/processed/bcn20000/` | `data/processed/bcn20000_cancer_risk/` |
| Metadata source | `bcn20000_metadata_2026-05-07.csv` | `bcn20000_metadata_2026-05-11.csv` |
| Classes | Melanoma / Benign nevus / Other lesion | Melanoma / Non-melanoma skin cancer / Benign nevus / Other non-cancer / indeterminate lesion |
| Image check | Required (drops rows if image missing) | Not required (image_path stored, checked at training time) |
| Usable rows | Varies by image availability | 17,639 (all rows with valid diagnosis_3) |

**Key difference:** BCC and SCC are now a separate `Non-melanoma skin cancer` class instead of being hidden inside `Other lesion`.

## Block 2 — 4-Class Label Mapping Logic

In [1]:
# The mapping function used in prepare_bcn20000_cancer_risk.py
def map_class_label(diagnosis: str) -> str:
    d = diagnosis.lower()
    if 'melanoma' in d:
        return 'Melanoma'
    if 'basal cell carcinoma' in d or 'squamous cell carcinoma' in d:
        return 'Non-melanoma skin cancer'
    if 'nevus' in d or 'nevi' in d:
        return 'Benign nevus'
    return 'Other non-cancer / indeterminate lesion'

# Verify mapping against all known diagnosis_3 values
test_cases = [
    ('Melanoma, NOS',                        'Melanoma'),
    ('Melanoma metastasis',                  'Melanoma'),
    ('Basal cell carcinoma',                 'Non-melanoma skin cancer'),
    ('Squamous cell carcinoma, NOS',         'Non-melanoma skin cancer'),
    ('Nevus',                                'Benign nevus'),
    ('Solar or actinic keratosis',           'Other non-cancer / indeterminate lesion'),
    ('Seborrheic keratosis',                 'Other non-cancer / indeterminate lesion'),
    ('Solar lentigo',                        'Other non-cancer / indeterminate lesion'),
    ('Scar',                                 'Other non-cancer / indeterminate lesion'),
    ('Dermatofibroma',                       'Other non-cancer / indeterminate lesion'),
]

print('=== MAPPING VERIFICATION ===')
all_pass = True
for diag, expected in test_cases:
    result = map_class_label(diag)
    status = 'PASS' if result == expected else 'FAIL'
    if status == 'FAIL':
        all_pass = False
    print(f'  [{status}] {diag:<40} → {result}')
print()
print('All mappings correct.' if all_pass else 'MAPPING ERRORS FOUND.')

=== MAPPING VERIFICATION ===
  [PASS] Melanoma, NOS                            → Melanoma
  [PASS] Melanoma metastasis                      → Melanoma
  [PASS] Basal cell carcinoma                     → Non-melanoma skin cancer
  [PASS] Squamous cell carcinoma, NOS             → Non-melanoma skin cancer
  [PASS] Nevus                                    → Benign nevus
  [PASS] Solar or actinic keratosis               → Other non-cancer / indeterminate lesion
  [PASS] Seborrheic keratosis                     → Other non-cancer / indeterminate lesion
  [PASS] Solar lentigo                            → Other non-cancer / indeterminate lesion
  [PASS] Scar                                     → Other non-cancer / indeterminate lesion
  [PASS] Dermatofibroma                           → Other non-cancer / indeterminate lesion

All mappings correct.


## Block 3 — Load and Verify Output Splits

In [2]:
import pandas as pd

base = '../../../data/processed/bcn20000_cancer_risk'

train = pd.read_csv(f'{base}/train.csv')
val   = pd.read_csv(f'{base}/val.csv')
test  = pd.read_csv(f'{base}/test.csv')
master = pd.read_csv(f'{base}/master_metadata.csv')

print('=== SPLIT SIZES ===')
print(f'  train          : {len(train):>6,} rows')
print(f'  val            : {len(val):>6,} rows')
print(f'  test           : {len(test):>6,} rows')
print(f'  master_metadata: {len(master):>6,} rows')
print(f'  train+val+test : {len(train)+len(val)+len(test):>6,} rows (should match master)')
print()
print('=== COLUMNS ===')
print(list(train.columns))

=== SPLIT SIZES ===
  train          : 12,352 rows
  val            :  2,628 rows
  test           :  2,659 rows
  master_metadata: 17,639 rows
  train+val+test : 17,639 rows (should match master)

=== COLUMNS ===
['isic_id', 'attribution', 'copyright_license', 'age_approx', 'anatom_site_1', 'anatom_site_2', 'anatom_site_general', 'anatom_site_special', 'concomitant_biopsy', 'diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_confirm_type', 'image_type', 'lesion_id', 'melanocytic', 'sex', 'class_label', 'image_path', 'split']


## Block 4 — Class Distribution

In [3]:
CLASS_ORDER = [
    'Melanoma',
    'Non-melanoma skin cancer',
    'Benign nevus',
    'Other non-cancer / indeterminate lesion',
]

total = len(master)

print(f'{"Class":<45} {"Total":>7}  {"Train":>7}  {"Val":>5}  {"Test":>5}')
print('-' * 85)
for cls in CLASS_ORDER:
    t = (master['class_label'] == cls).sum()
    tr = (train['class_label'] == cls).sum()
    v  = (val['class_label'] == cls).sum()
    te = (test['class_label'] == cls).sum()
    print(f'{cls:<45} {t:>7,}  {tr:>7,}  {v:>5,}  {te:>5,}')
print('-' * 85)
print(f'{"TOTAL":<45} {total:>7,}  {len(train):>7,}  {len(val):>5,}  {len(test):>5,}')
print()

cancer = master[master['class_label'].isin(['Melanoma', 'Non-melanoma skin cancer'])]
noncancer = master[~master['class_label'].isin(['Melanoma', 'Non-melanoma skin cancer'])]
print(f'Cancer / malignant (Melanoma + NMSC) : {len(cancer):,} ({len(cancer)/total*100:.1f}%)')
print(f'Non-cancer (Benign nevus + Other)    : {len(noncancer):,} ({len(noncancer)/total*100:.1f}%)')

Class                                           Total    Train    Val   Test
-------------------------------------------------------------------------------------
Melanoma                                        4,636    3,363    701    572
Non-melanoma skin cancer                        4,235    2,968    611    656
Benign nevus                                    5,647    3,934    778    935
Other non-cancer / indeterminate lesion         3,121    2,087    538    496
-------------------------------------------------------------------------------------
TOTAL                                          17,639   12,352  2,628  2,659

Cancer / malignant (Melanoma + NMSC) : 8,871 (50.3%)
Non-cancer (Benign nevus + Other)    : 8,768 (49.7%)


## Block 5 — Lesion-Level Leakage Check

In [4]:
train_lesions = set(train['lesion_id'])
val_lesions   = set(val['lesion_id'])
test_lesions  = set(test['lesion_id'])

tv_overlap  = train_lesions & val_lesions
tt_overlap  = train_lesions & test_lesions
vt_overlap  = val_lesions   & test_lesions

print('=== LEAKAGE CHECK ===')
print(f'  Unique lesions — train : {len(train_lesions):,}')
print(f'  Unique lesions — val   : {len(val_lesions):,}')
print(f'  Unique lesions — test  : {len(test_lesions):,}')
print()
print(f'  train ∩ val  overlap   : {len(tv_overlap)}  (must be 0)')
print(f'  train ∩ test overlap   : {len(tt_overlap)}  (must be 0)')
print(f'  val   ∩ test overlap   : {len(vt_overlap)}  (must be 0)')
print()
if tv_overlap or tt_overlap or vt_overlap:
    print('LEAKAGE DETECTED — splits are not safe.')
else:
    print('No leakage — splits are lesion-level safe.')

=== LEAKAGE CHECK ===
  Unique lesions — train : 3,520
  Unique lesions — val   : 754
  Unique lesions — test  : 755

  train ∩ val  overlap   : 0  (must be 0)
  train ∩ test overlap   : 0  (must be 0)
  val   ∩ test overlap   : 0  (must be 0)

No leakage — splits are lesion-level safe.


## Block 6 — Excluded Rows

In [5]:
raw = pd.read_csv('../../bcn20000_metadata_2026-05-11.csv')

excluded = raw[raw['diagnosis_3'].isna()]

print('=== EXCLUDED ROWS (missing diagnosis_3) ===')
print(f'  Total metadata rows   : {len(raw):,}')
print(f'  Excluded              : {len(excluded):,}')
print(f'  Usable (mapped)       : {len(raw) - len(excluded):,}')
print()
print('  diagnosis_1 distribution of excluded rows:')
for val_, count in excluded['diagnosis_1'].fillna('Missing').value_counts().items():
    print(f'    {val_:<40}: {count:,}')
print()
print('  These rows are not included in any split.')
print('  Reason: diagnosis_3 is the most specific label available.')
print('          Including ambiguous rows would add label noise.')

=== EXCLUDED ROWS (missing diagnosis_3) ===
  Total metadata rows   : 18,946
  Excluded              : 1,307
  Usable (mapped)       : 17,639

  diagnosis_1 distribution of excluded rows:
    Missing                                 : 1,156
    Benign                                  : 151

  These rows are not included in any split.
  Reason: diagnosis_3 is the most specific label available.
          Including ambiguous rows would add label noise.


## Block 7 — Summary

| Item | Detail |
|------|--------|
| Issue | #118 — D3.3 |
| Status | Closed |
| Taxonomy | 4-class cancer-risk (approved #117) |
| Metadata source | `bcn20000_metadata_2026-05-11.csv` |
| Split ratio | 70 / 15 / 15 (seed=42) |
| Split key | `lesion_id` — leakage-safe |
| Usable rows | 17,639 |
| Excluded rows | 1,307 (missing `diagnosis_3`) |
| Old splits | Untouched — `data/processed/bcn20000/` |

### Dependency chain
```
D3.1 — Approve cancer-risk taxonomy    ✅ (#116)
  → D3.2 (#117) — Create label mapping  ✅
    → D3.3 (#118) — Rebuild splits       ✅ THIS NOTEBOOK
      → D3.4 (#119) — Retrain CNN
        → D3.5 (#120) — Evaluate CNN
```

### Next step
#119 — Retrain dermoscopic CNN using `data/processed/bcn20000_cancer_risk/train.csv` and `val.csv`.